In [1]:
import torch
import numpy as np

import time

from botorch.models.gp_regression import SingleTaskGP
from botorch.models.deterministic import GenericDeterministicModel
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.models.model import ModelList

from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch import fit_gpytorch_mll
from botorch.optim.optimize import optimize_acqf
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.utils.multi_objective.box_decompositions.dominated import DominatedPartitioning

import warnings
from botorch.exceptions.warnings import NumericsWarning
warnings.filterwarnings("ignore", category=NumericsWarning)
warnings.filterwarnings("ignore", category = RuntimeWarning)

tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

torch.random.fork_rng(devices = [0])

# Define default parameters for the BO loop
REFERENCE = torch.tensor([-0.1, 0.0], **tkwargs)
BATCH_SIZE = 10
N_ITER = 10
MAX_CONC = 8
DIM = 7
BOUNDS = torch.tensor([[0] * DIM, [MAX_CONC] * DIM], **tkwargs)

# Define default hyperparams for the acqf optimization routine
DEFAULT_PARAMS = {
    "raw_samples": 256,
    "num_restarts": 10,
    "batch_limit": 5,
    "max_iter": 100
}


In [2]:
from mlp import MLP

# Load the surrogate model
mdl = MLP().to(**tkwargs)
mdl.load_state_dict(torch.load('models/surrogate.pth', weights_only=True))
mdl.eval()

# Define a function to query the surrogate model
def surrogate_model(X):
    return mdl(X).detach()

# Define a function to query the deterministic model
def deterministic_model(X):
    return X.sum(dim=-1).unsqueeze(-1).detach()

In [3]:
def loadData():
    data = np.concatenate([
        np.load('data/data_initial.npy'),
        np.load('data/data_iter1.npy'),
        np.load('data/data_iter2.npy')
    ])
    return torch.Tensor(data).to(**tkwargs)

def initialize_model(train_X, train_Y):
    # Define deterministic model (on total concentration, doesnt require train_Y though)
    deterministic_model = GenericDeterministicModel(lambda x: x.sum(dim=-1, keepdim=True))
    # Define GP model (on viability)
    gp_model = SingleTaskGP(
        train_X,
        train_Y[:,1].unsqueeze(-1),
        input_transform = Normalize(train_X.shape[-1]),
        outcome_transform = Standardize(m=1)
    )
    model = ModelList(deterministic_model, gp_model)
    mll = ExactMarginalLogLikelihood(model.models[1].likelihood, model.models[1])
    return mll, model

def outcome_constraint(Z: torch.Tensor) -> torch.Tensor:
    return Z[..., 0] - MAX_CONC

def step_mobo(model, train_X, raw_samples, num_restarts, batch_limit, max_iter):
    acq = qLogNoisyExpectedHypervolumeImprovement(
        model = model,
        ref_point = REFERENCE,
        X_baseline = train_X,
        constraints = [outcome_constraint],
        prune_baseline = True
    )
    candidates, _ = optimize_acqf(
        acq_function = acq,
        bounds = BOUNDS,
        q = BATCH_SIZE,
        num_restarts = num_restarts,
        raw_samples = raw_samples,
        options = {
            "batch_limit": batch_limit,
            "maxiter": max_iter
        },
        sequential = True
    )
    return candidates

def run_bo():
    init_X = loadData()[:,:-2]
    init_Y = loadData()[:,-2:]

    mll, model = initialize_model(init_X, init_Y)

    train_X, train_Y = init_X.clone(), init_Y.clone()

    hvs = [DominatedPartitioning(REFERENCE, init_Y).compute_hypervolume().item()]
    times = [0.0]
    candidates_list = [init_X.cpu().tolist()]

    for i in range(1, N_ITER+1):
        start = time.time()
        fit_gpytorch_mll(mll)
        candidates = step_mobo(model, train_X, **DEFAULT_PARAMS)
        end = time.time()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Compute new_X and new_Y
        new_X = candidates.detach()
        new_Y = torch.cat([deterministic_model(new_X), surrogate_model(new_X)], dim=-1)

        train_X = torch.cat([train_X, new_X], dim=0)
        train_Y = torch.cat([train_Y, new_Y], dim=0)

        hv = DominatedPartitioning(ref_point=REFERENCE, Y = train_Y).compute_hypervolume().item()

        # Save all relevant data to lists
        hvs.append(hv)
        times.append(end-start)
        candidates_list.append(candidates.cpu().tolist())

        mll, model = initialize_model(train_X, train_Y)

        print(f"Iteration {i} | HV = {hv:.5f}, t = {end-start:.2f}s")

    return hvs, times, candidates_list

In [4]:
with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

/home/dan/Research/cpa-bayesian-opt/py31/lib/python3.11/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 4 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


Iteration 1 | HV = 659.40673, t = 64.61s
Iteration 2 | HV = 691.30672, t = 55.86s
Iteration 3 | HV = 700.61163, t = 59.33s
Iteration 4 | HV = 700.65988, t = 62.81s
Iteration 5 | HV = 708.54855, t = 88.74s
Iteration 6 | HV = 713.09024, t = 57.33s
Iteration 7 | HV = 713.80140, t = 74.04s
Iteration 8 | HV = 713.80334, t = 74.54s
Iteration 9 | HV = 713.90768, t = 78.25s
Iteration 10 | HV = 726.72052, t = 75.57s


In [5]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import os

from botorch.utils.multi_objective.box_decompositions.non_dominated import FastNondominatedPartitioning

def multi_objective(candidates):
    return torch.cat([deterministic_model(candidates), surrogate_model(candidates)], dim=-1)

def generate_pareto_gif(candidates_list, hvs, filename='pareto_front.gif'):
    images = []
    all_candidates = []
    all_pareto_Y = []

    for i, candidates in enumerate(candidates_list):
        # Get the function values for the current candidates
        # Define a function to query the surrogate model
        Y = multi_objective(torch.tensor(candidates, **tkwargs))

        # Append current candidates to all_candidates
        all_candidates.append(Y)

        # Compute the Pareto front
        combined_Y = torch.cat(all_candidates, dim=0)
        partitioning = FastNondominatedPartitioning(ref_point=REFERENCE, Y=combined_Y)
        pareto_Y = partitioning.pareto_Y

        # Append current Pareto front to all_pareto_Y
        all_pareto_Y.append(pareto_Y)

        # Plot the Pareto front
        plt.figure()
        for j, past_Y in enumerate(all_candidates):
            plt.scatter(past_Y[:, 0].cpu(), past_Y[:, 1].cpu(), color = 'black', alpha = 0.5)
        plt.scatter(pareto_Y[:, 0].cpu(), pareto_Y[:, 1].cpu(), color='red', label='Pareto Front')
        plt.title(f'Iteration {i}, Hypervolume: {hvs[i]:.5f}')
        plt.xlabel('Total Concentration')
        plt.ylabel('% Viability')
        # plt.xlim([0,MAX_CONC])
        plt.grid()

        # Save the plot to a temporary file
        temp_filename = f'temp_{i}.png'
        plt.savefig(temp_filename)
        plt.close()

        # Read the temporary file and append to the images list
        images.append(imageio.imread(temp_filename))

    # Create the GIF
    imageio.mimsave(filename, images, duration=1e3)

    # Clean up temporary files
    for i in range(len(candidates_list)):
        temp_filename = f'temp_{i}.png'
        os.remove(temp_filename)

    print(f'GIF saved as {filename}')

# Example usage
generate_pareto_gif(candidates_list, hvs)

GIF saved as pareto_front.gif


In [87]:
import matplotlib.pyplot as plt
from matplotlib import colormaps

def plot_composition(samples, legend = True, title = None, ax = None, filename = None):
    samples_names = [f'CPA {i+1}' for i in range(samples.shape[0])][::-1]
    feature_names = ['Glycerol', 'DMSO', 'EG', '12PD', '13PD', '3M12PD', 'Urea']
    feature_colors = colormaps.get_cmap('tab10').colors[:samples.shape[1]]

    if ax is None:
        fig = plt.figure()
        ax = fig.gca()

    left = torch.zeros(samples.shape[0])

    for i in range(samples.shape[1]):
        ax.barh(samples_names, samples[:, i].cpu(), color = feature_colors[i], left = left, alpha = 0.75)
        left += samples[:, i].cpu()

    if legend:
        ax.legend(feature_names, bbox_to_anchor=(1.0, 0.7))

    maximum = 8
    step = 0.5

    ax.set_xticks(np.arange(0, maximum + step, 2*step))
    ax.set_xticks(np.arange(step, maximum, 2*step), minor = True)
    ax.grid(axis='x', which = 'major', linestyle='--')
    # ax.grid(axis='x', which = 'minor', linestyle='--')

    ax.set_xlabel('Concentration (M)')
    
    ax.set_title(title)

    if (filename is None) & (ax is None):
        fig.show()

def plot_pareto(all_Y, indices = False, ax = None, filename = None):
    if ax is None:
        fig = plt.figure()
        ax = fig.gca()

    partitioning = FastNondominatedPartitioning(ref_point=REFERENCE, Y=all_Y)
    pareto_Y = partitioning.pareto_Y

    hv = partitioning.compute_hypervolume().item()

    ax.scatter(all_Y[:, 0].cpu(), all_Y[:, 1].cpu(), color = 'black', alpha = 0.5)
    ax.scatter(pareto_Y[:, 0].cpu(), pareto_Y[:, 1].cpu(), color='red', label='Pareto Front')
    ax.set_xlabel('Total Concentration')
    ax.set_ylabel('% Viability')
    ax.set_title(f'Hypervolume: {hv:.5f}')
    ax.grid()

    if (filename is None) & (ax is None):
        fig.show()

    if indices:
        pareto_indices = []
        partitioning = FastNondominatedPartitioning(ref_point=REFERENCE, Y=all_Y)
        pareto_Y = partitioning.pareto_Y
        for point in pareto_Y:
            # Find the row index in `data` where the last two columns match the Pareto point
            indices = torch.where((all_Y[:,0] == point[0]) & (all_Y[:,1] == point[1]))[0]
            for index in indices:
                pareto_indices.append(index.item())
        return pareto_indices

In [97]:
def plot_iteration(all_X, all_Y, candidates = None, title = None, filename = None):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    # Plot the Pareto front
    indices = plot_pareto(all_Y, indices = True, ax = ax[0])

    # Plot the composition of the Pareto front
    plot_composition(all_X[indices], legend = False, title = 'Pareto Front Composition', ax = ax[1])

    # For all but the zeroth iteration plot the composition of the candidates
    if candidates is not None:
        plot_composition(candidates, legend = True, title = 'Candidate Composition', ax = ax[2])

    if title:
        fig.suptitle(title)

    fig.tight_layout()

    if filename is None:
        plt.show()
    else:
        plt.savefig(filename)
        plt.close()

In [95]:
def generate_gif(candidates_list, filename='composite.gif'):
    images = []
    Y_list = []
    X_list = []

    for i, candidates in enumerate(candidates_list):
        # Get the function values for the current candidates
        Y = multi_objective(torch.tensor(candidates, **tkwargs))

        Y_list.append(Y)
        all_Y = torch.cat(Y_list, dim=0)

        X_list.append(torch.tensor(candidates, **tkwargs))
        all_X = torch.cat(X_list, dim=0)

        # Plot the iteration
        if i == 0:
            plot_iteration(all_X, all_Y, title = f'Iteration #{i}', filename = f'temp{i}.png')
        else:
            plot_iteration(all_X, all_Y, candidates = torch.tensor(candidates, **tkwargs), title = f'Iteration #{i}', filename = f'temp{i}.png')

        # Read the temporary file and append to the images list
        images.append(imageio.imread(f'temp{i}.png'))

        # Clean up temporary files
        os.remove(f'temp{i}.png')

    # Create the GIF
    imageio.mimsave(filename, images, duration=1e3)

    print(f'GIF saved as {filename}')

In [98]:
generate_gif(candidates_list)

GIF saved as composite.gif
